In [3]:
import pandas as pd
import requests
import time
import os
import zipfile
import urllib.request
from tqdm import tqdm 
from dotenv import load_dotenv, find_dotenv


In [2]:

# Tự động tìm file .env ở các thư mục cha
load_dotenv(find_dotenv())

# Sử dụng biến bình thường
TMDB_API_KEY = os.getenv("TMDB_API_KEY_THANG")
print(TMDB_API_KEY)

b9e1a9bec7021432f58a97f4b2a00f97


In [3]:
DATA_PATH = "../data"

In [ ]:
# !pip install tqdm

In [ ]:
# !pip3 install kagglehub 

In [ ]:
# !pip install kagglehub[hf-datasets]

In [4]:
import kagglehub




MOVIELENS_DIR = os.path.join(DATA_PATH, 'movielens')
path = ""
if not os.path.exists(MOVIELENS_DIR):
    # Download latest version
    path = kagglehub.dataset_download("grouplens/movielens-latest-full")
else:
    path = MOVIELENS_DIR
    
print("Path to dataset files:", path)

Path to dataset files: ../data\movielens


c:\Users\THUYEN\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
full_df = {
    "ratings": pd.read_csv(path + "/ratings.csv"),
    "tags": pd.read_csv(path + "/tags.csv"),
    "links": pd.read_csv(path + "/links.csv"),
    "movies": pd.read_csv(path + "/movies.csv"),
    "genome_scores": pd.read_csv(path + "/genome_scores.csv"),
    "genome_tags": pd.read_csv(path + "/genome_tags.csv")
}

In [6]:
print('Số phim: ', full_df['movies'].shape[0]) 
print('Số người dùng: ', full_df['ratings']['userId'].nunique())
print('Số lượt đánh giá: ', full_df['ratings'].shape[0])

Số phim:  58098
Số người dùng:  283228
Số lượt đánh giá:  27753444


In [7]:
for name, df in full_df.items():
    print(f"{50*'='}")
    print(f"DataFrame '{name}':")
    print(df.head())
    
    save_path = os.path.join(MOVIELENS_DIR, f"{name}.csv")
    df.to_csv(save_path, index=False)
    print("\n")

DataFrame 'ratings':
   userId  movieId  rating   timestamp
0       1      307     3.5  1256677221
1       1      481     3.5  1256677456
2       1     1091     1.5  1256677471
3       1     1257     4.5  1256677460
4       1     1449     4.5  1256677264


DataFrame 'tags':
   userId  movieId           tag   timestamp
0      14      110          epic  1443148538
1      14      110      Medieval  1443148532
2      14      260        sci-fi  1442169410
3      14      260  space action  1442169421
4      14      318  imdb top 250  1442615195


DataFrame 'links':
   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0


DataFrame 'movies':
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (199

In [8]:
# 1. Nhóm theo movieId để tính Số lượt đánh giá và Điểm trung bình
movie_stats = full_df['ratings'].groupby('movieId').agg(
    rating_count=('rating', 'count'),
    rating_mean=('rating', 'mean')
).reset_index()

# 2. Áp dụng tiêu chuẩn lọc (Filter)
MIN_RATINGS = 10  # Phim phải có ít nhất 50 người rate
MIN_SCORE = 2.0   # Điểm trung bình từ 3.0 trở lên

quality_movies = movie_stats[
    (movie_stats['rating_count'] >= MIN_RATINGS) & 
    (movie_stats['rating_mean'] >= MIN_SCORE)
]

print(f"Số phim ban đầu: {len(movie_stats)}")
print(f"Số phim 'Đáng xem' sau khi lọc: {len(quality_movies)}")

# 3. Gộp (Merge) kết quả đã lọc với bảng links_df để lấy tmdbId
filtered_links_df = pd.merge(quality_movies, full_df['links'], on='movieId', how='inner')

display(filtered_links_df.sort_values(by='rating_count', ascending=False).head())

Số phim ban đầu: 53889
Số phim 'Đáng xem' sau khi lọc: 23226


,movieId,rating_count,rating_mean,imdbId,tmdbId
314,318,97999,4.424188,111161,278.0
351,356,97040,4.056585,109830,13.0
292,296,92406,4.173971,110912,680.0
585,593,87899,4.151412,102926,274.0
2439,2571,84545,4.149695,133093,603.0


In [9]:
MIN_RELEASE_YEAR = 1995

def fetch_movie_vietnamese_metadata(tmdb_id):
    """Hàm gọi API TMDB để lấy thông tin phim bằng tiếng Việt"""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=vi-VN"
    
    try:
        response = requests.get(url, timeout=10)
        # Nếu request thành công
        if response.status_code == 200:
            data = response.json()
            
            # check date sau 1995
            if 'release_date' in data and data['release_date']:
                release_year = int(data['release_date'].split('-')[0])
                if release_year < MIN_RELEASE_YEAR:
                    return None
            is_fetch_again = False
            # Nếu không có tiếng việt, gọi lại với ngôn ngữ mặc định (tiếng Anh)
            if not data.get('overview') or data.get('overview') == '':
                url_en = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=en-US"
                response_en = requests.get(url_en, timeout=10)
                if response_en.status_code == 200:
                    data_en = response_en.json()
                    # data['title'] = data_en.get('title', '')
                    data['overview'] = data_en.get('overview', '')
                    is_fetch_again = True
            
            return {
                'tmdbId': tmdb_id,
                'title_vi': data.get('title', ''),
                'overview_vi' :is_fetch_again == False and data.get('overview', '') or '', # Nếu trống thì sẽ dịch ở giai đoạn sau
                # Lấy danh sách tên thể loại và nối thành chuỗi
                'genres': ", ".join([g['name'] for g in data.get('genres', [])]) if data.get('genres') else "",
                'release_date': data.get('release_date', ''),
                'poster_path': data.get('poster_path', '') # Rất hữu ích cho frontend sau này,
                
            } | data
        # Nếu phim không tồn tại trên TMDB
        elif response.status_code == 404:
            return None
    except Exception as e:
        print(f"Lỗi khi fetch tmdbId {tmdb_id}: {e}")
        return None
    return None

In [ ]:
tmdb_id = 91550 
metadata = fetch_movie_vietnamese_metadata(tmdb_id)
for k, v in metadata.items():
    print(f"{k}: {v}")


In [10]:
movie_vi_data = []

# Nơi lưu phim
TMDB_DIR = os.path.join(DATA_PATH, 'tmdb')
if not os.path.exists(TMDB_DIR):
    os.makedirs(TMDB_DIR)
    
# Kiểm tra đã lưu trước hay chưa
if os.path.exists(os.path.join(TMDB_DIR, "movies_vietnamese_metadata.jsonl")):
    print("Dữ liệu phim tiếng Việt đã tồn tại, không cần fetch lại.")
    movie_vi_data = pd.read_json(os.path.join(TMDB_DIR, "movies_vietnamese_metadata.jsonl"), lines=True).to_dict(orient='records')

# fetch theo đoạn: limit, offset
last_offset = 10000
batch_size = 1000


def fetch_movie_batch(limit=1000, offset=0):
    """Hàm fetch một batch phim từ filtered_links_df"""
    batch_links = filtered_links_df.iloc[offset:offset+limit]
    batch_data = [] 
    
    for index, row in tqdm(batch_links.iterrows(), total=batch_links.shape[0], desc=f"Đang fetch batch {offset + 1}-{offset + limit}"):
        movie_id = row['movieId']
        tmdb_id = row['tmdbId']
        
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)
        
        if metadata:
            metadata['movieId'] = movie_id
            batch_data.append(metadata)
        
        time.sleep(0.05)  # Tránh bị block API
    
    movie_vi_data.extend(batch_data)
    save_movie(batch_data, offset, limit)
    return batch_data, offset + limit

def save_movie(data, offset, limit, prefix="movies_vietnamese_metadata", is_all=False):
    # Chuyển list các dictionary thành Pandas DataFrame
    movies_vi_df = pd.DataFrame(data)

    # Hiển thị vài dòng đầu để kiểm tra
    # display(movies_vi_df.head())

        
    start_index = offset + 1
    end_index = offset + len(data)
    if is_all:
        output_filename = os.path.join(TMDB_DIR, f"{prefix}.jsonl")
    else:
        output_filename = os.path.join(TMDB_DIR, f"{prefix}_{start_index}-{end_index}.jsonl")
    movies_vi_df.to_json(output_filename, orient='records', lines=True, force_ascii=False)
    print(f"\nĐã lưu thành công {len(movies_vi_df)} phim tiếng Việt vào file {output_filename}!")




In [11]:
for i in range(10):
    data, last_offset = fetch_movie_batch(limit=batch_size, offset=last_offset)

Đang fetch batch 10001-11000: 100%|██████████| 1000/1000 [47:04<00:00,  2.82s/it] 



Đã lưu thành công 663 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_10001-10663.jsonl!


Đang fetch batch 11001-12000:  18%|█▊        | 185/1000 [09:41<1:22:00,  6.04s/it]

Lỗi khi fetch tmdbId 6535.0: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Read timed out. (read timeout=10)


Đang fetch batch 11001-12000: 100%|██████████| 1000/1000 [36:09<00:00,  2.17s/it] 


...


Đã lưu thành công 812 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_18001-18812.jsonl!


Đang fetch batch 19001-20000:  46%|████▌     | 458/1000 [19:10<50:36,  5.60s/it]

Lỗi khi fetch tmdbId 21175.0: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Read timed out. (read timeout=10)


Đang fetch batch 19001-20000: 100%|██████████| 1000/1000 [38:58<00:00,  2.34s/it]


Đã lưu thành công 863 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_19001-19863.jsonl!


In [12]:
save_movie(movie_vi_data, offset=0, limit=len(movie_vi_data), prefix="movies_vietnamese_metadata_10000", is_all=True)


Đã lưu thành công 7720 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_10000.jsonl!


In [13]:
movie_vi_data_df = pd.DataFrame(movie_vi_data)

In [22]:
movie_vi_data_df.to_csv(os.path.join(TMDB_DIR, "movies_metadata_11000-20000_1995-01-01.csv"), index=False, encoding='utf-8-sig')

In [14]:
movie_vi_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7720 entries, 0 to 7719
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tmdbId                 7720 non-null   float64
 1   title_vi               7720 non-null   str    
 2   overview_vi            7720 non-null   str    
 3   genres                 7720 non-null   object 
 4   release_date           7720 non-null   str    
 5   poster_path            7712 non-null   str    
 6   adult                  7720 non-null   bool   
 7   backdrop_path          7538 non-null   str    
 8   belongs_to_collection  1312 non-null   object 
 9   budget                 7720 non-null   int64  
 10  homepage               7720 non-null   str    
 11  id                     7720 non-null   int64  
 12  imdb_id                7720 non-null   str    
 13  origin_country         7720 non-null   object 
 14  original_language      7720 non-null   str    
 15  original_title 

In [15]:
movie_vi_data_df = movie_vi_data_df.dropna(subset=['overview_vi'])

# 2. Tính trung bình số từ (length) của cột overview_vi
mean_length = movie_vi_data_df['overview_vi'].str.split().str.len().mean()
print("Trung bình length của overview_vi:", mean_length)


Trung bình length của overview_vi: 21.936658031088083


In [20]:
# 3. Lọc các phim có overview ngắn hơn 50 từ
short_movies_df = movie_vi_data_df[movie_vi_data_df['overview_vi'].str.split().str.len() == 0]
display(short_movies_df.sort_values('release_date', ascending=False).head(10))



# Lọc phim năm 1995 trở đi
recent_movies_df = movie_vi_data_df[movie_vi_data_df['release_date'] >= '2000-01-01']
display(recent_movies_df.sort_values('release_date', ascending=True).head(100))

,tmdbId,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,movieId
7347,216527.0,Avatar 4,,"[{'id': 878, 'name': 'Phim Khoa Học Viễn Tưởng...",2029-12-19,/qzMYKnT4MG1d0gnhwytr4cKhUvS.jpg,False,NaN,"{'id': 87096, 'name': 'Loạt phim Avatar', 'pos...",0,...,0,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Planned,,Avatar 4,False,0.000,0,135448.0
6662,321130.0,Солярис,,"[{'id': 10770, 'name': 'Chương Trình Truyền Hì...",2019-04-14,/82D3HjM3UJRMpmyxoGqjjotRenp.jpg,False,/kr9d1AX9D82AjKMsW00t00FXFVn.jpg,None,0,...,142,False,"[{'english_name': 'Russian', 'iso_639_1': 'ru'...",Released,,Солярис,False,6.000,13,126426.0
7699,300667.0,3 Generations,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",2016-10-27,/g4kR79ndKP6gZFHXe70sY7TUOZ7.jpg,False,/aefR29q5J8RrtDS5k0dFEdKmzl4.jpg,None,10000000,...,87,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,3 Generations,False,6.357,448,140291.0
7379,293452.0,Căn Phòng Ám Ảnh,,"[{'id': 27, 'name': 'Phim Kinh Dị'}, {'id': 53...",2016-09-09,/vv2LgcbuV9omqxlFBpOQEb6VkyT.jpg,False,/4Xu5gaD3bZwN8w7qQklPo7bfjiI.jpg,None,15000000,...,92,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Đừng đánh thức cơn ác mộng kinh hoàng!,Căn Phòng Ám Ảnh,False,4.509,484,135643.0
6700,310119.0,Last Days in the Desert,,"[{'id': 12, 'name': 'Phim Phiêu Lưu'}, {'id': ...",2016-05-13,/7vbP28axycrpchnGSIydUseNkh6.jpg,False,/vivLzGFFDEAg9iXc61g2ogswiRg.jpg,None,0,...,98,False,"[{'english_name': 'German', 'iso_639_1': 'de',...",Released,,Last Days in the Desert,False,5.731,182,127122.0
7113,321030.0,One More Time,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",2016-04-08,/yrZ1azTISRD91tcYCeMk7lhhYgc.jpg,False,/2FQnGgZb8E94xl9fUWspnJZB4xj.jpg,None,0,...,98,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,One More Time,False,5.035,72,132658.0
7086,333089.0,Bad Hurt,,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2016-02-12,/77Cea2Hdjmncy1BSre1lkCo2098.jpg,False,/rxNICwg18ggBZ0jj8eEUSz9sWh5.jpg,None,0,...,100,False,[],Released,,Bad Hurt,False,5.200,17,132520.0
7664,318781.0,Colonia Dignidad – Es gibt kein Zurück,,"[{'id': 18, 'name': 'Phim Chính Kịch'}, {'id':...",2016-02-12,/D9Zu8QS1SnELUKNupwkyA55Fd3.jpg,False,/6gtlv8NYhzEeRyfbIbfeXp5KVJR.jpg,None,14000000,...,106,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Colonia Dignidad – Es gibt kein Zurück,False,7.254,2020,139857.0
5351,194188.0,Punk's Dead: SLC Punk 2,,"[{'id': 35, 'name': 'Phim Hài'}]",2016-02-11,/AptQQxD4HMNMZZQuVD1XKojCJiZ.jpg,False,/eaadGOWEwKa0nIK6QaP3s1jpYvR.jpg,"{'id': 592064, 'name': 'SLC Punk Collection', ...",0,...,75,False,[],Released,,Punk's Dead: SLC Punk 2,False,4.500,29,107516.0
6722,308453.0,The Bronze,,"[{'id': 18, 'name': 'Phim Chính Kịch'}, {'id':...",2016-01-22,/Ah2nD3Kml59svErVsz6DUsxlJ8.jpg,False,/57b4FmA9aSNE6wiL439DlKFS5TY.jpg,None,3500000,...,108,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Bronze,False,6.000,404,127192.0


,tmdbId,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,movieId
1381,41245.0,박하사탕,,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2000-01-01,/zxCr6MBF7BPr4r8RiYyuNps4DWN.jpg,False,/d9LNlmhjihHAAP1vKCQJo57JQFX.jpg,None,0,...,131,False,"[{'english_name': 'Korean', 'iso_639_1': 'ko',...",Released,,박하사탕,False,7.506,309,61018.0
1892,20152.0,Englar alheimsins,,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2000-01-01,/7ylhuBS1aBmZXgoOwCWPwcaWwNs.jpg,False,/vP5hmJoc0zbbBElSFKw5PsgqTNV.jpg,None,0,...,100,False,"[{'english_name': 'Icelandic', 'iso_639_1': 'i...",Released,,Englar alheimsins,False,7.200,34,69464.0
4728,40878.0,The Crossing,,"[{'id': 10770, 'name': 'Chương Trình Truyền Hì...",2000-01-10,/9LuskivFVMBaUf1RFfqUwE8KNwG.jpg,False,/sV5bKNU246dhWWI8zVFVarxGKe5.jpg,None,0,...,89,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Crossing,False,6.404,26,101322.0
3044,16987.0,कहो ना प्यार है,,"[{'id': 10749, 'name': 'Phim Lãng Mạn'}, {'id'...",2000-01-14,/alALytPcaZyQkzOjnRgHMSRv8fq.jpg,False,/maXxt6GuXk67NTjtqlbo04xCtf3.jpg,None,0,...,172,False,"[{'english_name': 'Hindi', 'iso_639_1': 'hi', ...",Released,,कहो ना प्यार है,False,6.704,164,83266.0
7335,124134.0,"Up, Up, and Away",,"[{'id': 28, 'name': 'Phim Hành Động'}, {'id': ...",2000-01-22,/kKBOr6r4EcCNFC2seWYbFn9mZWj.jpg,False,/1GKHwfn66E6kWMJkBD158Y8VyKT.jpg,None,0,...,77,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,"Up, Up, and Away",False,5.500,55,135270.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2763,49943.0,Сёстры,,"[{'id': 80, 'name': 'Phim Hình Sự'}, {'id': 18...",2001-05-10,/3d20lZgTCkOCkHzMHv9DWSFS6eE.jpg,False,/1BciwSCWd9EF0NaH8bAs3zvwNNw.jpg,None,0,...,83,False,"[{'english_name': 'Russian', 'iso_639_1': 'ru'...",Released,,Сёстры,False,6.800,96,80124.0
1627,10596.0,Kẻ Thế Thân,Cảnh sát Seattle bất lực trong việc truy bắt t...,"[{'id': 28, 'name': 'Phim Hành Động'}, {'id': ...",2001-05-11,/q41zGgRXfhvnObJnn1kGIyoAhSu.jpg,False,/hmLfKkw8adwLvS89SOTYY88ZpN9.jpg,None,17000000,...,100,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Kẻ Thế Thân,False,5.854,550,65552.0
7540,37789.0,R.O.D リード・オア・ダイ,,"[{'id': 16, 'name': 'Phim Hoạt Hình'}, {'id': ...",2001-05-23,/zF1ZFrCAW56M5bN6izUIsy6AShg.jpg,False,/wnjsP6SuyJRPr1OwhwDXLFw3cv8.jpg,None,0,...,90,False,"[{'english_name': 'Japanese', 'iso_639_1': 'ja...",Released,,R.O.D リード・オア・ダイ,True,6.037,27,136956.0
5410,38189.0,The Follow,,"[{'id': 28, 'name': 'Phim Hành Động'}, {'id': ...",2001-05-24,/wAVZSrnlRQsNDrmwqIatzXLDjsN.jpg,False,/hgqcKNJb4iJRdzVMbOkEmdbfuRy.jpg,"{'id': 113589, 'name': 'The Hire', 'poster_pat...",0,...,11,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Follow,False,6.853,95,108095.0


In [19]:
recent_movies_df.count()

tmdbId                   7474
title_vi                 7474
overview_vi              7474
genres                   7474
release_date             7474
poster_path              7467
adult                    7474
backdrop_path            7302
belongs_to_collection    1244
budget                   7474
homepage                 7474
id                       7474
imdb_id                  7474
origin_country           7474
original_language        7474
original_title           7474
overview                 7474
popularity               7474
production_companies     7474
production_countries     7474
revenue                  7474
runtime                  7474
softcore                 7474
spoken_languages         7474
status                   7474
tagline                  7474
title                    7474
video                    7474
vote_average             7474
vote_count               7474
movieId                  7474
dtype: int64